In [ ]:
# Importação das bibliotecas necessárias
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import random

# Semente para reprodutibilidade dos resultados
np.random.seed(1234)
random.seed(1234)


In [ ]:
# Função para calcular a superfície de erro J(a1, a2)
def calculateErrorSurface(y, x1, x2):
    M = 200  # Número de pontos no grid
    a1 = np.linspace(-12.0, 14.0, M)
    a2 = np.linspace(-12.0, 14.0, M)
    A1, A2 = np.meshgrid(a1, a2)
    J = np.zeros((M, M))

    for i in range(M):
        for j in range(M):
            yhat = A1[i, j]*x1 + A2[i, j]*x2
            J[i, j] = (1.0/N)*np.sum((y - yhat)**2)

    return J, A1, A2


In [ ]:
# Função para implementar o Gradiente Descendente Estocástico (SGD)
def gradientStochasticDescent(X, y, n_epochs, alpha):
    N = len(y)
    a = np.array([-10.0, -10.0]).reshape(2, 1)  # pesos iniciais
    a_hist = np.zeros((2, n_epochs*N+1))
    grad_hist = np.zeros((2, n_epochs*N))
    Jgd = np.zeros(n_epochs*N+1)

    a_hist[:, 0] = a.ravel()
    Jgd[0] = (1.0/N)*np.sum((y - X.dot(a))**2)

    for epoch in range(n_epochs):
        shuffled_indexes = random.sample(range(N), N)

        for i in range(N):
            idx = shuffled_indexes[i]
            xi = X[idx:idx+1]
            yi = y[idx:idx+1]

            gradient = -2.0 * xi.T.dot(yi - xi.dot(a))
            a -= alpha * gradient

            k = epoch*N + i
            a_hist[:, k+1] = a.ravel()
            Jgd[k+1] = (1.0/N)*np.sum((y - X.dot(a))**2)
            grad_hist[:, k] = gradient.ravel()

    return a, Jgd, a_hist, grad_hist


In [ ]:
# Geração dos dados sintéticos
N = 1000
x1 = np.random.randn(N, 1)
x2 = np.random.randn(N, 1)
y = 2.0 * x1 + 3.0 * x2
w = np.random.randn(N, 1)
y_noisy = y + w

# Montagem da matriz de atributos (sem bias)
X = np.c_[x1, x2]


In [ ]:
# Solução ótima (Equação Normal)
a_opt = np.linalg.pinv(X.T.dot(X)).dot(X.T.dot(y_noisy))
yhat = X.dot(a_opt)
Joptimum = (1.0/N)*np.sum((y_noisy - yhat)**2)

print(f"Solução ótima:")
print(f"a1 = {a_opt[0, 0]:.4f}, a2 = {a_opt[1, 0]:.4f}")
print(f"Erro ótimo (MSE) = {Joptimum:.6f}")


In [ ]:
# Visualização da função de custo
J, A1, A2 = calculateErrorSurface(y_noisy, x1, x2)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(A1, A2, J, cmap=cm.coolwarm)
ax.set_xlabel("a1")
ax.set_ylabel("a2")
ax.set_zlabel("Erro")
ax.set_title("Superfície da Função de Custo")
plt.show()


In [ ]:
# Execução do SGD
n_epochs = 5
alpha = 0.1
a_sgd, Jgd, a_hist, grad_hist = gradientStochasticDescent(X, y_noisy, n_epochs, alpha)


In [ ]:
# Contorno da função de custo e trajetória do SGD
plt.figure(figsize=(6, 5))
cp = plt.contour(A1, A2, J)
plt.clabel(cp)
plt.plot(a_opt[0], a_opt[1], 'r*', markersize=12, label='Solução ótima')
plt.plot(a_hist[0], a_hist[1], 'kx', markersize=2, label='SGD')
plt.xlabel("a1")
plt.ylabel("a2")
plt.title("Caminho do SGD na Função de Custo")
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Curva de convergência do erro ao longo das iterações
plt.figure(figsize=(6, 4))
plt.plot(np.arange(len(Jgd)), Jgd)
plt.yscale("log")
plt.xlabel("Iteração")
plt.ylabel("Erro (MSE)")
plt.title("Erro vs Iteração (SGD)")
plt.grid()
plt.show()


In [ ]:
# Evolução dos gradientes por componente
plt.figure(figsize=(6, 4))
plt.plot(grad_hist[0], label="∂J/∂a1")
plt.plot(grad_hist[1], label="∂J/∂a2")
plt.xlabel("Iteração")
plt.ylabel("Gradiente")
plt.title("Gradientes ao longo das iterações")
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Função para Gradiente Descendente em Mini-Batch
def gradientMiniBatchDescent(X, y, n_epochs, alpha, batch_size):
    N = len(y)
    a = np.array([-10.0, -10.0]).reshape(2, 1)  # inicialização dos pesos
    total_iterations = n_epochs * (N // batch_size)
    
    a_hist = np.zeros((2, total_iterations + 1))
    grad_hist = np.zeros((2, total_iterations))
    Jgd = np.zeros(total_iterations + 1)

    a_hist[:, 0] = a.ravel()
    Jgd[0] = (1.0/N) * np.sum((y - X.dot(a))**2)

    iteration = 0
    for epoch in range(n_epochs):
        # embaralha os dados
        shuffled_indices = np.random.permutation(N)
        X_shuffled = X[shuffled_indices]
        y_shuffled = y[shuffled_indices]

        for i in range(0, N, batch_size):
            X_batch = X_shuffled[i:i + batch_size]
            y_batch = y_shuffled[i:i + batch_size]

            # gradiente para o mini-lote
            gradient = -2.0 * X_batch.T.dot(y_batch - X_batch.dot(a)) / batch_size
            a -= alpha * gradient

            # salva histórico
            iteration += 1
            a_hist[:, iteration] = a.ravel()
            Jgd[iteration] = (1.0/N) * np.sum((y - X.dot(a))**2)
            grad_hist[:, iteration-1] = gradient.ravel()

    return a, Jgd, a_hist, grad_hist
